In [ ]:
# ============================================================
# OPENFLEXURE AUTOFOCUS — CANCER TISSUE EXPERIMENT
# RESULTS PLOTTING
# ============================================================
#
# Generates:
#   1. Initial Variance of Laplacian
#   2. Final Variance of Laplacian
#   3. Autofocus Time
#   4. CPU Usage
#   5. RAM Usage
#   6. Coefficient of Variation (CV)
#   7. Relative Error
#
# Methods:
#   - SVR
#   - Random Forest
#   - Custom AF / Modified VoL
#   - Inbuilt AF
#
# Cancer tissue tested at:
#   - 500 steps
#   - 1000 steps
#   - 1500 steps
#
# ============================================================
# EQUATIONS
# ============================================================
#
# RELATIVE ERROR:
#
#        Relative Error = |Z_true - Z_predicted|
#                         ---------------------- × 100%
#                                |Z_true|
#
#
# COEFFICIENT OF VARIATION:
#
#        CV = SD(Z_final)
#             ----------- × 100
#             |Mean(Z_final)|
#
# ============================================================


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# 1. USER SETTINGS
# ============================================================

FILES = {

    500: "500 defocus cancer.csv",

    1000: "1000 defocus cancer.csv",

    1500: "1500 defocus cancer.csv"

}


# ============================================================
# OUTPUT FOLDER
# ============================================================

OUTPUT_FOLDER = "cancer_autofocus_results 2"


# ============================================================
# 2. METHOD NAMES
# ============================================================

METHOD_LABELS = {

    "SVR": "SVR",

    "RANDOM FOREST": "Random Forest",

    "CUSTOM": "Custom AF",

    "INBUILT": "Inbuilt AF"

}


METHOD_ORDER = [

    "SVR",

    "RANDOM FOREST",

    "CUSTOM",

    "INBUILT"

]


# ============================================================
# 3. CREATE OUTPUT FOLDER
# ============================================================

output_folder = Path(
    OUTPUT_FOLDER
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. FUNCTION TO READ CSV FILE
# ============================================================

def read_autofocus_csv(filename):

    print("\n")
    print("=" * 70)
    print("READING FILE:")
    print(filename)
    print("=" * 70)

    raw = pd.read_csv(
        filename,
        header=None
    )

    methods = {}

    current_method = None


    # ========================================================
    # READ EACH ROW
    # ========================================================

    for _, row in raw.iterrows():

        first_value = (

            str(row.iloc[0]).strip()

            if pd.notna(row.iloc[0])

            else ""

        )


        # ====================================================
        # DETECT METHOD
        # ====================================================

        if first_value.upper() in {

            "SVR",

            "RANDOM FOREST",

            "CUSTOM",

            "INBUILT"

        }:

            current_method = (
                first_value.upper()
            )

            methods[current_method] = []

            continue


        # ====================================================
        # READ DATA FOR CURRENT METHOD
        # ====================================================

        if (

            current_method is not None

            and not pd.isna(row.iloc[0])

        ):

            methods[current_method].append(

                row.iloc[:7].tolist()

            )


    # ========================================================
    # COLUMN NAMES
    # ========================================================

    columns = [

        "Z_Initial",

        "Z_Final",

        "Initial_Variance",

        "Final_Variance",

        "Autofocus_Time_s",

        "CPU",

        "RAM"

    ]


    # ========================================================
    # CREATE DATAFRAME FOR EACH METHOD
    # ========================================================

    result = {}


    for method, rows in methods.items():

        df = pd.DataFrame(

            rows,

            columns=columns

        )


        # ====================================================
        # CONVERT DATA TO NUMERIC
        # ====================================================

        for column in columns:

            df[column] = pd.to_numeric(

                df[column],

                errors="coerce"

            )


        # ====================================================
        # REMOVE INVALID Z VALUES
        # ====================================================

        df = df.dropna(

            subset=[

                "Z_Initial",

                "Z_Final"

            ]

        ).reset_index(

            drop=True

        )


        result[method] = df


    return result


# ============================================================
# 5. LOAD CANCER DATA
# ============================================================

all_data = {}


for defocus, filename in FILES.items():

    all_data[defocus] = (

        read_autofocus_csv(
            filename
        )

    )


# ============================================================
# 6. CALCULATE RESULTS
# ============================================================

results = []


for defocus, methods in all_data.items():

    print("\n")
    print("-" * 70)

    print(
        f"PROCESSING CANCER TISSUE — "
        f"{defocus} STEPS"
    )

    print("-" * 70)


    # ========================================================
    # CHECK CUSTOM AF
    # ========================================================

    if "CUSTOM" not in methods:

        raise ValueError(

            f"Custom AF data was not found "
            f"in the {defocus}-step CSV file."

        )


    # ========================================================
    # GROUND TRUTH
    # ========================================================

    z_true = (

        methods["CUSTOM"]
        ["Z_Final"]
        .mean()

    )


    print(

        f"Z_true = "
        f"{z_true:.3f} steps"

    )


    # ========================================================
    # PROCESS EACH METHOD
    # ========================================================

    for method in METHOD_ORDER:

        if method not in methods:

            print(

                f"WARNING: {method} "
                f"not found at {defocus} steps."

            )

            continue


        df = methods[method]


        z_predicted = (

            df["Z_Final"]
            .to_numpy()

        )


        absolute_error = np.abs(

            z_true
            - z_predicted

        )


        # ====================================================
        # RELATIVE ERROR
        # ====================================================

        rel_err_each_trial = (

            absolute_error

            / abs(z_true)

            * 100

        )


        relative_error = (

            rel_err_each_trial.mean()

        )


        # ====================================================
        # COEFFICIENT OF VARIATION
        # ====================================================

        z_mean = (

            df["Z_Final"]
            .mean()

        )


        z_sd = (

            df["Z_Final"]
            .std(

                ddof=1

            )

        )


        cv = (

            z_sd

            / abs(z_mean)

            * 100

        )


        # ====================================================
        # INITIAL VARIANCE
        # ====================================================

        initial_variance_mean = (

            df["Initial_Variance"]
            .mean()

        )


        initial_variance_sd = (

            df["Initial_Variance"]
            .std(

                ddof=1

            )

        )


        # ====================================================
        # FINAL VARIANCE
        # ====================================================

        final_variance_mean = (

            df["Final_Variance"]
            .mean()

        )


        final_variance_sd = (

            df["Final_Variance"]
            .std(

                ddof=1

            )

        )


        # ====================================================
        # AUTOFOCUS TIME
        # ====================================================

        autofocus_time_mean = (

            df["Autofocus_Time_s"]
            .mean()

        )


        autofocus_time_sd = (

            df["Autofocus_Time_s"]
            .std(

                ddof=1

            )

        )


        # ====================================================
        # CPU
        # ====================================================

        cpu_mean = (

            df["CPU"]
            .mean()

        )


        cpu_sd = (

            df["CPU"]
            .std(

                ddof=1

            )

        )


        # ====================================================
        # RAM
        # ====================================================

        ram_mean = (

            df["RAM"]
            .mean()

        )


        # ====================================================
        # STORE RESULTS
        # ====================================================

        results.append({

            "Defocus_steps":
                defocus,

            "Method":
                METHOD_LABELS[method],

            "N":
                len(df),

            "Z_true_steps":
                z_true,

            "Initial_Variance_Mean":
                initial_variance_mean,

            "Initial_Variance_SD":
                initial_variance_sd,

            "Final_Variance_Mean":
                final_variance_mean,

            "Final_Variance_SD":
                final_variance_sd,

            "Autofocus_Time_Mean_s":
                autofocus_time_mean,

            "Autofocus_Time_SD_s":
                autofocus_time_sd,

            "CPU_Mean_percent":
                cpu_mean,

            "CPU_SD_percent":
                cpu_sd,

            "RAM_Mean_MB":
                ram_mean,

            "CV_percent":
                cv,

            "Relative_Error_percent":
                relative_error

        })


# ============================================================
# 7. CREATE SUMMARY DATAFRAME
# ============================================================

summary = pd.DataFrame(

    results

)


# ============================================================
# 8. SAVE CSV SUMMARY
# ============================================================

summary_csv = (

    output_folder

    / "cancer_autofocus_results_summary.csv"

)


summary.to_csv(

    summary_csv,

    index=False

)


# ============================================================
# 9. SAVE EXCEL SUMMARY
# ============================================================

summary_excel = (

    output_folder

    / "cancer_autofocus_results_summary.xlsx"

)


with pd.ExcelWriter(

    summary_excel,

    engine="openpyxl"

) as writer:

    summary.to_excel(

        writer,

        sheet_name="Cancer Results",

        index=False

    )


# ============================================================
# 10. PLOTTING FUNCTION (600 DPI)
# ============================================================

def create_plot(

    metric,

    error_metric,

    title,

    ylabel,

    filename,

    decimals=2

):


    # ========================================================
    # FIGURE (600 DPI)
    # ========================================================

    fig, ax = plt.subplots(

        figsize=(12, 7),

        dpi=600

    )


    # ========================================================
    # X POSITIONS
    # ========================================================

    x = np.arange(3)


    # ========================================================
    # BAR WIDTH
    # ========================================================

    width = 0.18


    # ========================================================
    # PLOT METHODS
    # ========================================================

    for i, method_key in enumerate(

        METHOD_ORDER

    ):

        method_name = (

            METHOD_LABELS[
                method_key
            ]

        )


        data = (

            summary[
                summary["Method"]
                == method_name
            ]

            .sort_values(
                "Defocus_steps"
            )

        )


        values = (

            data[metric]
            .to_numpy()

        )


        if error_metric is not None:

            errors = (

                data[
                    error_metric
                ]

                .to_numpy()

            )

        else:

            errors = None


        bars = ax.bar(

            x

            + (i - 1.5)
            * width,

            values,

            width,

            yerr=errors,

            capsize=5

            if errors is not None

            else 0,

            label=method_name,

            edgecolor="black",

            linewidth=0.8

        )


        # ====================================================
        # MEAN VALUE LABELS (Proportional offset for small values)
        # ====================================================

        for bar, value in zip(

            bars,

            values

        ):

            if np.isfinite(value):

                offset = 0.03 * abs(value) + 0.05

                ax.text(

                    bar.get_x()
                    + bar.get_width()
                    / 2,

                    bar.get_height()
                    + offset,

                    f"{value:.{decimals}f}",

                    ha="center",

                    va="bottom",

                    fontsize=10,

                    fontweight="bold"

                )


    # ========================================================
    # X AXIS
    # ========================================================

    ax.set_xticks(x)


    ax.set_xticklabels(

        [

            "500",

            "1000",

            "1500"

        ],

        fontsize=13,

        fontweight="bold"

    )


    ax.set_xlabel(

        "Initial defocus (motor steps)",

        fontsize=14,

        fontweight="bold"

    )


    # ========================================================
    # Y AXIS
    # ========================================================

    ax.set_ylabel(

        ylabel,

        fontsize=14,

        fontweight="bold"

    )


    # ========================================================
    # TITLE
    # ========================================================

    ax.set_title(

        title,

        fontsize=16,

        fontweight="bold"

    )


    ax.tick_params(

        axis="y",

        labelsize=12

    )


    # ========================================================
    # GRID
    # ========================================================

    ax.grid(

        axis="y",

        linestyle="--",

        alpha=0.35

    )


    # ========================================================
    # REMOVE TOP / RIGHT SPINES
    # ========================================================

    ax.spines[
        "top"
    ].set_visible(False)


    ax.spines[
        "right"
    ].set_visible(False)


    # ========================================================
    # LEGEND
    # ========================================================

    ax.legend(

        title="Autofocus method",

        fontsize=11,

        title_fontsize=11,

        loc="best"

    )


    # ========================================================
    # LAYOUT
    # ========================================================

    fig.tight_layout()


    # ========================================================
    # SAVE (600 DPI)
    # ========================================================

    output_file = (

        output_folder

        / filename

    )


    fig.savefig(

        output_file,

        dpi=600,

        bbox_inches="tight"

    )


    plt.close(fig)


    print(

        f"Saved: {output_file}"

    )


# ============================================================
# 11. GENERATE THE SEVEN GRAPHS
# ============================================================


# ============================================================
# GRAPH 1 — INITIAL VARIANCE
# ============================================================

create_plot(

    metric=
        "Initial_Variance_Mean",

    error_metric=
        "Initial_Variance_SD",

    title=
        "Initial Image Sharpness — Cancer Tissue",

    ylabel=
        "Initial Variance of Laplacian",

    filename=
        "01_cancer_initial_variance.png",

    decimals=2

)


# ============================================================
# GRAPH 2 — FINAL VARIANCE
# ============================================================

create_plot(

    metric=
        "Final_Variance_Mean",

    error_metric=
        "Final_Variance_SD",

    title=
        "Final Image Sharpness — Cancer Tissue",

    ylabel=
        "Final Variance of Laplacian",

    filename=
        "02_cancer_final_variance.png",

    decimals=2

)


# ============================================================
# GRAPH 3 — AUTOFOCUS TIME
# ============================================================

create_plot(

    metric=
        "Autofocus_Time_Mean_s",

    error_metric=
        "Autofocus_Time_SD_s",

    title=
        "Autofocus Execution Time — Cancer Tissue",

    ylabel=
        "Autofocus time (s)",

    filename=
        "03_cancer_autofocus_time.png",

    decimals=2

)


# ============================================================
# GRAPH 4 — CPU
# ============================================================

create_plot(

    metric=
        "CPU_Mean_percent",

    error_metric=
        "CPU_SD_percent",

    title=
        "CPU Utilisation — Cancer Tissue",

    ylabel=
        "CPU usage (%)",

    filename=
        "04_cancer_cpu_usage.png",

    decimals=2

)


# ============================================================
# GRAPH 5 — RAM
# ============================================================

create_plot(

    metric=
        "RAM_Mean_MB",

    error_metric=None,

    title=
        "RAM Utilisation — Cancer Tissue",

    ylabel=
        "RAM usage (MB)",

    filename=
        "05_cancer_ram_usage.png",

    decimals=2

)


# ============================================================
# GRAPH 6 — CV
# ============================================================

create_plot(

    metric=
        "CV_percent",

    error_metric=None,

    title=
        "Autofocus Repeatability — Cancer Tissue",

    ylabel=
        "Coefficient of variation (%)",

    filename=
        "06_cancer_cv_percent.png",

    decimals=2

)


# ============================================================
# GRAPH 7 — RELATIVE ERROR
# ============================================================

create_plot(

    metric=
        "Relative_Error_percent",

    error_metric=None,

    title=
        "Relative Error — Cancer Tissue",

    ylabel=
        "Relative error (%)",

    filename=
        "07_cancer_relative_error.png",

    decimals=2

)


# ============================================================
# 12. DISPLAY FINAL RESULTS
# ============================================================

print("\n")

print("=" * 70)

print(
    "CANCER TISSUE AUTOFOCUS EXPERIMENT SUMMARY"
)

print("=" * 70)


print(

    summary[

        [

            "Defocus_steps",

            "Method",

            "N",

            "Z_true_steps",

            "Relative_Error_percent",

            "CV_percent"

        ]

    ].round(3)

)


# ============================================================
# 13. PRINT OUTPUT LOCATION
# ============================================================

print("\n")

print("=" * 70)

print("FILES SAVED")

print("=" * 70)


print(

    "\nOutput folder:\n",

    output_folder.resolve()

)


print(

    "\nSummary CSV:\n",

    summary_csv.resolve()

)


print(

    "\nSummary Excel:\n",

    summary_excel.resolve()

)


print("\n")

print(
    "All seven cancer-tissue graphs "
    "have been generated successfully at 600 DPI."
)

print("\n")

print("=" * 70)